In [4]:
# Standard library imports
import sys
from pathlib import Path

# Third-party imports
import duckdb
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import seaborn as sns

# Add src to path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

# Display settings
pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')
%matplotlib inline

## 1. Initialize Data Connection (DuckDB)

**Optimization:** Using DuckDB to query the Parquet file directly.

In [5]:
# Define path to dataset
dashboard_path = Path('../data/processed/dashboard_data.parquet')

if not dashboard_path.exists():
    raise FileNotFoundError(
        "Dashboard dataset not found! Please run notebook 06_feature_engineering.ipynb first."
    )

print("Initializing DuckDB connection...")
con = duckdb.connect(database=':memory:')

# Create a view
con.execute(f"CREATE OR REPLACE VIEW crashes AS SELECT * FROM '{dashboard_path}'")

# Inspect columns to find the correct date column name
columns_info = con.execute("DESCRIBE crashes").fetchall()
columns = [col[0] for col in columns_info]
print(f"Available columns: {columns}")

# Helper to find column case-insensitively
def find_col(candidates):
    for cand in candidates:
        if cand in columns:
            return f'"{cand}"'
    return None

date_col = find_col(['CRASH DATE', 'CRASH_DATE', 'crash_date'])
if not date_col:
    raise ValueError("Could not find crash date column!")

print(f"Using date column: {date_col}")

# Get basic stats
total_records = con.execute("SELECT COUNT(*) FROM crashes").fetchone()[0]
date_range = con.execute(f"SELECT MIN({date_col}), MAX({date_col}) FROM crashes").fetchone()

print(f"✓ Connected to dataset")
print(f"✓ Total Records: {total_records:,}")
print(f"✓ Date Range: {date_range[0]} to {date_range[1]}")

Initializing DuckDB connection...
Available columns: ['COLLISION_ID', 'CRASH DATE', 'CRASH TIME', 'BOROUGH', 'ZIP CODE', 'LATITUDE', 'LONGITUDE', 'NUMBER OF PERSONS INJURED', 'NUMBER OF PERSONS KILLED', 'NUMBER OF PEDESTRIANS INJURED', 'NUMBER OF PEDESTRIANS KILLED', 'NUMBER OF CYCLIST INJURED', 'NUMBER OF CYCLIST KILLED', 'NUMBER OF MOTORIST INJURED', 'NUMBER OF MOTORIST KILLED', 'CONTRIBUTING FACTOR VEHICLE 1', 'VEHICLE TYPE CODE 1', 'PERSON_TYPE', 'PERSON_AGE', 'PERSON_SEX', 'PERSON_INJURY', 'PED_ROLE', 'COMPLAINT', 'BODILY_INJURY', 'POSITION_IN_VEHICLE', 'crash_year', 'crash_month', 'crash_day', 'crash_day_of_week', 'crash_day_name', 'crash_quarter', 'crash_week_of_year', 'crash_time_obj', 'crash_hour', 'season', 'borough_clean', 'has_borough', 'has_coordinates', 'total_injured', 'total_killed', 'total_casualties', 'severity_score', 'severity_category', 'is_high_severity', 'pedestrians_injured', 'pedestrians_killed', 'involves_pedestrian', 'cyclists_injured', 'cyclists_killed', 'in

## 2. Temporal Analysis

Analyze crash trends over time.

In [6]:
# 2.1 Yearly Trend
query_year = """
    SELECT crash_year, COUNT(*) as crashes 
    FROM crashes 
    WHERE crash_year IS NOT NULL
    GROUP BY 1 
    ORDER BY 1
"""
df_year = con.execute(query_year).df()

fig_year = px.line(df_year, x='crash_year', y='crashes', 
                   title='Total Crashes per Year',
                   markers=True)
fig_year.show()

In [7]:
# 2.2 Hourly Trend
query_hour = """
    SELECT crash_hour, COUNT(*) as crashes 
    FROM crashes 
    WHERE crash_hour IS NOT NULL
    GROUP BY 1 
    ORDER BY 1
"""
df_hour = con.execute(query_hour).df()

fig_hour = px.bar(df_hour, x='crash_hour', y='crashes', 
                  title='Crashes by Hour of Day',
                  color='crashes')
fig_hour.show()

## 3. Location Analysis

Analyze crashes by Borough.

In [8]:
# 3.1 Crashes by Borough
query_borough = """
    SELECT borough_clean, COUNT(*) as crashes 
    FROM crashes 
    WHERE borough_clean IS NOT NULL
    GROUP BY 1 
    ORDER BY 2 DESC
"""
df_borough = con.execute(query_borough).df()

fig_borough = px.bar(df_borough, x='borough_clean', y='crashes', 
                     title='Total Crashes by Borough',
                     color='borough_clean')
fig_borough.show()

In [9]:
# 3.2 High Severity Map (Top 1000)
# We need to find the correct latitude/longitude columns too
lat_col = find_col(['LATITUDE', 'latitude'])
lon_col = find_col(['LONGITUDE', 'longitude'])

if lat_col and lon_col:
    query_map = f"""
        SELECT {lat_col} as lat, {lon_col} as lon, total_casualties, borough_clean
        FROM crashes 
        WHERE has_coordinates = TRUE 
          AND is_high_severity = TRUE
        ORDER BY total_casualties DESC
        LIMIT 1000
    """
    df_map = con.execute(query_map).df()

    fig_map = px.scatter_mapbox(df_map, lat="lat", lon="lon", 
                                color="borough_clean", size="total_casualties",
                                zoom=10, height=600,
                                title="Top 1000 High Severity Crashes")
    fig_map.update_layout(mapbox_style="open-street-map")
    fig_map.show()
else:
    print("Latitude/Longitude columns not found for mapping.")

C:\Users\Mohamed\AppData\Local\Temp\ipykernel_27848\1996464358.py:17: DeprecationWarning:

*scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/



## 4. Severity & Contributing Factors

Analyze what causes crashes and how severe they are.

In [10]:
# 4.1 Severity Distribution
query_sev = """
    SELECT severity_category, COUNT(*) as crashes 
    FROM crashes 
    WHERE severity_category IS NOT NULL
    GROUP BY 1
"""
df_sev = con.execute(query_sev).df()

fig_sev = px.pie(df_sev, values='crashes', names='severity_category', 
                 title='Distribution of Crash Severity')
fig_sev.show()

In [11]:
# 4.2 Top Contributing Factors
# Find the correct column for contributing factor
factor_col = find_col(['CONTRIBUTING FACTOR VEHICLE 1', 'CONTRIBUTING_FACTOR_VEHICLE_1'])

if factor_col:
    query_factor = f"""
        SELECT {factor_col} as factor, COUNT(*) as crashes 
        FROM crashes 
        WHERE {factor_col} IS NOT NULL 
          AND {factor_col} != 'Unspecified'
        GROUP BY 1 
        ORDER BY 2 DESC 
        LIMIT 10
    """
    df_factor = con.execute(query_factor).df()

    fig_factor = px.bar(df_factor, y='factor', x='crashes', 
                        title='Top 10 Contributing Factors',
                        orientation='h')
    fig_factor.update_layout(yaxis={'categoryorder':'total ascending'})
    fig_factor.show()
else:
    print("Contributing factor column not found.")

In [ ]:
# Close connection
con.close()
print("Analysis complete.")

Analysis complete.


: 